កំណត់ត្រាទាំងនេះត្រូវបានបង្កើតដោយស្វ័យប្រវត្តិដោយ GitHub Copilot Chat ហើយមានគោលបំណងសម្រាប់ការតម្លើងដំបូងតែប៉ុណ្ណោះ


# ការណែនាំអំពីវិស្វកម្មពាក្យបញ្ជា
វិស្វកម្មពាក្យបញ្ជា គឺជាដំណើរការរចនា និងបង្កើនប្រសិទ្ធភាពនៃពាក្យបញ្ជាសម្រាប់បេសកកម្មបំលែងភាសារូបិយបច្ចេកវិទ្យា។ វារួមមានការជ្រើសរើសពាក្យបញ្ជាដែលត្រឹមត្រូវ ការបញ្ជ្រាបប៉ារ៉ាម៉ែត្រ និងការវាយតម្លៃលទ្ធផលរបស់វា។ វិស្វកម្មពាក្យបញ្ជាមានសារៈសំខាន់សម្រាប់ការទទួលបានភាពត្រឹមត្រូវខ្ពស់ និងប្រសិទ្ធភាពល្អក្នុងម៉ូដែល NLP។ នៅក្នុងផ្នែកនេះ យើងនឹងសិក្សាអំពីមូលដ្ឋាននៃវិស្វកម្មពាក្យបញ្ជាដោយប្រើម៉ូដែល OpenAI សម្រាប់ការស្វែងយល់។


### លំហាត់ ១៖ ការចែកពាក្យ
ស្វែងយល់អំពីការចែកពាក្យដោយប្រើ tiktoken ដែលជាប្រព័ន្ធចែកពាក្យលឿនបើកឡើងពី OpenAI
សូមមើល [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) សម្រាប់ឧទាហរណ៍បន្ថែម។


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### ស្រោចស្រង់ 2៖ ផ្ទៀងផ្ទាត់ការតំឡើងកូនសោ API របស់ OpenAI

រត់កូដខាងក្រោមដើម្បីផ្ទៀងផ្ទាត់ថាប្រសិនបើចំណុចបញ្ចូល OpenAI របស់អ្នកត្រូវបានតំឡើងឲ្យត្រឹមត្រូវ។ កូដនេះគ្រាន់តែព្យាយាមប្រើប្រាស់កំហូបមូលដ្ឋានសាមញ្ញមួយ ហើយផ្ទៀងផ្ទាត់ការសម្រេចបាន។ អញ្ញាតិ `oh say can you see` គួរតែបញ្ចប់ក្នុងបែប `by the dawn's early light..`


In [ ]:
# Uses the OpenAI client against the Azure OpenAI (Microsoft Foundry) v1 endpoint
# with the Responses API. See https://aka.ms/openai/start

import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']

def get_completion(prompt):
    response = client.responses.create(
        model=deployment,
        input=prompt,
        temperature=0, # this is the degree of randomness of the model's output
        max_output_tokens=1024,
        store=False,
    )
    return response.output_text

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt)
print(response)


### ការធ្វើហ تمر 3: ការបង្កើតក្លែងក្លាយ
ស្វែងរកអ្វីដែលកើតឡើងនៅពេលដែលអ្នកសួរឲ្យ LLM ឆ្លើយតបនូវការបញ្ចប់សម្រាប់ការបញ្ជូនអំពីប្រធានបទដែលអាចមិនមាន ឬអំពីប្រធានបទដែលវាអាចមិនស្គាល់ព្រោះវាបាននៅក្រៅសំណុំទិន្នន័យដែលវាត្រូវបានបណ្តុះបណ្តាលជាមុន (ថ្មីជាងនេះ)។ មើលថាតើការឆ្លើយតបផ្លាស់ប្តូរយ៉ាងដូចម្តេច ប្រសិនបើអ្នកសាកល្បងការបញ្ជូនផ្សេង ឬម៉ូដែលផ្សេង។


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt)
print(response)

### លំហាត់ ៤: អាស្រ័យលើការណែនាំ  
ប្រើអថេរ "text" ដើម្បីកំណត់មាតិកា​ចម្បង  
និងអថេរ "prompt" ដើម្បីផ្តល់ការណែនាំដែលទាក់ទងនឹងមាតិកាចម្បងនោះ។  

នៅទីនេះយើងសុំឲ្យម៉ូដែលសង្ខេបអត្ថបទសម្រាប់សិស្សថ្នាក់ទីពីរ  


In [ ]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt)
print(response)

### ធ្វើលំហាត់ ៥: សំណើពិបាក 
សូមសាកល្បងសំណើដែលមានសារពីប្រព័ន្ធ អ្នកប្រើ បញ្ជំនិងជំនួយការ 
ប្រព័ន្ធកំណត់បរិបទជំនួយការ
សារអ្នកប្រើ និងជំនួយការផ្ដល់បរិបទការសន្ទនាច្រើនជំហាន

សូមយកចិត្តទុកដាក់ពីរបៀបដែលបុគ្គលិកភាពជំនួយការត្រូវបានកំណត់ជាប្រភេទ "អុកកក្បាស់" នៅក្នុងបរិបទប្រព័ន្ធ។ 
សាកល្បងប្រើបុគ្គលិកភាពផ្សេងទៀត ឬសាកល្បងផ្នែកសារបញ្ជូល / ផលិតបញ្ចេញផ្សេងទៀត។


In [ ]:
response = client.responses.create(
    model=deployment,
    input=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ],
    store=False,
)
print(response.output_text)


### អនុវត្ត៖ ស្វែងយល់អារម្មណ៍របស់អ្នក
ឧទាហរណ៍ខាងលើផ្តល់ឲ្យអ្នកនូវគំរូដែលអ្នកអាចប្រើសម្រាប់បង្កើតការបញ្ចូលថ្មីៗ (សាមញ្ញ, ស្មុគស្មាញ, ណែនាំ ល។) - សាកល្បងបង្កើតអនុវត្តន៍ផ្សេងទៀតដើម្បីស្វែងយល់ពីគំនិតផ្សេងទៀតដែលយើងបានពិភាក្សាដូចជា ឧទាហរណ៍, គន្លឹះ និងផ្សេងទៀត។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
